# Analisi polarimetrica — tesi triennale

Notebook ordinato che orchestra la pipeline 2D (Stokes + retardance + UMAP) usando il package `polarimetro/`.

**Ordine pipeline obbligato** (vedi `CLAUDE.md`):
1. `reset_saturation_accumulator()`
2. `load_rotation_sequence(...)` → `calculate_linear_stokes`
3. `calculate_s3(...)` (popola `_WAV_INTENSITY_CACHE` per il rebasing Poincaré)
4. `generate_background_mask(S0)`
5. `align_reference_frame` → `align_poincare_ellipticity` (riassegnare S1, S3)
6. `calculate_dolp_aolp` + `calculate_retardance_and_fast_axis(target_folder=...)`

## Configurazione globale

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import polarimetro as pol
from polarimetro import config as polcfg
from polarimetro import plotting as polplot
from polarimetro import umap_runner as polumap

polplot.apply_thesis_style()

DATASET = 'strati_v2'
CHANNEL = 'all'  # 'R' | 'G' | 'B' | 'all'
DOWNSAMPLE_FACTOR = 4
SAVE_PLOTS = True
OUTPUT_DIR = '../Images/generated'

RUN_SPECTRA = False
RUN_DEBUGGER = False
RUN_FULL_BATCH = False

TARGET_FOLDER = f'./raw/{DATASET}'
POL_SUBFOLDER = os.path.join(TARGET_FOLDER, 'pol')
WAV_SUBFOLDER = os.path.join(TARGET_FOLDER, 'wav')
WAVELENGTHS_CSV = './outputs/rgb_wavelengths.csv'
DARK_FRAME_PATH = './raw/dark.dng'

CHANNELS_RGB = {'R': 0, 'G': 1, 'B': 2}
ACTIVE_CHANNELS = list(CHANNELS_RGB.keys()) if CHANNEL == 'all' else [CHANNEL]

print(f"dataset={DATASET}, channels={ACTIVE_CHANNELS}, DS={DOWNSAMPLE_FACTOR}, swap={polcfg.is_waveplate_swapped(TARGET_FOLDER)}")

## Dispatcher analisi per dataset

Per ciascun dataset elenca le analisi da eseguire. Le celle stub sotto controllano `DATASET in dispatcher[...]` per gated execution.

In [ ]:
ANALYSES_PER_DATASET = {
    'strati_v2':         ['AB', 'C-delta', 'F', 'H', 'E-ext'],
    'lambdaquarti_50deg': ['AB', 'C-delta'],
    'lambdamezzi_50deg':  ['AB', 'C-delta'],
    'zucchero':          ['AB', 'C-aolp'],
    'barraon_v2':        ['AB'],
    'barraoff_v2':       ['AB'],
    'righello_v2':       ['AB'],
}

active_analyses = ANALYSES_PER_DATASET.get(DATASET, [])
print(f"analisi attive per {DATASET}: {active_analyses}")

## Load + Stokes — pipeline completa con cache npz

Per ogni canale attivo: carica RAW, calcola Stokes lineari, S3, maschere, applica allineamento (asse S3 + asse S2 / Poincaré), deriva DoLP/AoLP/δ/θ. Cache `outputs/stokes_<DATASET>_DS<DS>.npz` invalidata se cambia `DOWNSAMPLE_FACTOR` o `DATASET`.

In [ ]:
print(f"=== Load+Stokes: DATASET='{DATASET}', DS={DOWNSAMPLE_FACTOR}, channels={ACTIVE_CHANNELS} ===")
print(f"    cache_target = ./outputs/stokes_{DATASET}_DS{DOWNSAMPLE_FACTOR}.npz")

stokes_data = {}

if 'AB' in active_analyses:
    os.makedirs('./outputs', exist_ok=True)
    cache_path = f'./outputs/stokes_{DATASET}_DS{DOWNSAMPLE_FACTOR}.npz'

    cache_valid = False
    if os.path.exists(cache_path):
        try:
            npz = np.load(cache_path, allow_pickle=False)
            if int(npz['downsample_factor']) == DOWNSAMPLE_FACTOR \
                    and str(npz['dataset']) == DATASET \
                    and all(f'{ch}_S0' in npz.files for ch in ACTIVE_CHANNELS):
                for ch in ACTIVE_CHANNELS:
                    stokes_data[ch] = {
                        'S0': npz[f'{ch}_S0'],
                        'S1': npz[f'{ch}_S1'],
                        'S2': npz[f'{ch}_S2'],
                        'S3': npz[f'{ch}_S3'],
                        'bg_mask': npz[f'{ch}_bg_mask'].astype(bool),
                        'poincare_bg_mask': npz[f'{ch}_poincare_bg_mask'].astype(bool),
                        'sat_mask': (npz[f'{ch}_sat_mask'].astype(bool)
                                     if f'{ch}_sat_mask' in npz.files else None),
                        'DoLP': npz[f'{ch}_DoLP'],
                        'AoLP': npz[f'{ch}_AoLP'],
                        'delta': npz[f'{ch}_delta'],
                        'theta': npz[f'{ch}_theta'],
                        'wav_intensity': (npz[f'{ch}_wav_intensity']
                                          if f'{ch}_wav_intensity' in npz.files else None),
                        'channel_idx': CHANNELS_RGB[ch],
                    }
                cache_valid = True
                print(f"Cache loaded: {cache_path}  (npz.dataset='{str(npz[\"dataset\"])}', npz.DS={int(npz[\"downsample_factor\"])}, channels={list(ACTIVE_CHANNELS)})")
            npz.close()
        except Exception as e:
            print(f"Cache read failed ({e}); recomputing.")

    if not cache_valid:
        for ch in ACTIVE_CHANNELS:
            ch_idx = CHANNELS_RGB[ch]
            print(f"\n=== Channel {ch} (idx={ch_idx}) ===")

            pol.reset_all_caches()  # sat + wav_intensity + poincare bg mask

            angles, stack = pol.load_rotation_sequence(
                POL_SUBFOLDER, ch_idx,
                downsample_factor=DOWNSAMPLE_FACTOR,
                invert_angles=True,
                dark_frame_path=DARK_FRAME_PATH,
            )
            if angles is None or stack is None:
                raise RuntimeError(f"load_rotation_sequence failed for {DATASET}/{ch}")

            S0, S1, S2 = pol.calculate_linear_stokes(angles, stack)

            wavelength = polcfg.get_channel_wavelength(WAVELENGTHS_CSV, ch_idx)
            S3 = pol.calculate_s3(
                WAV_SUBFOLDER, ch_idx,
                downsample_factor=DOWNSAMPLE_FACTOR,
                wavelength=wavelength,
                dark_frame_path=DARK_FRAME_PATH,
            )

            bg_mask = pol.generate_background_mask(S0, downsample_factor=DOWNSAMPLE_FACTOR)
            S1, S2 = pol.align_reference_frame(S1, S2, bg_mask)
            S1, S3 = pol.align_poincare_ellipticity(
                S0, S1, S3, bg_mask,
                downsample_factor=DOWNSAMPLE_FACTOR,
            )

            DoLP, AoLP = pol.calculate_dolp_aolp(S0, S1, S2)
            delta_deg, theta_deg = pol.calculate_retardance_and_fast_axis(
                S0, S1, S2, S3, bg_mask,
                target_folder=TARGET_FOLDER,
            )

            sat_mask = pol.get_saturation_mask(DOWNSAMPLE_FACTOR)
            if sat_mask is not None:
                n_sat = int(sat_mask.sum())
                if n_sat:
                    frac = 100.0 * n_sat / sat_mask.size
                    print(f"Saturation: {n_sat} clipped blocks ({frac:.2f}%)")
                    for arr in (S1, S2, S3, DoLP, AoLP, delta_deg, theta_deg):
                        arr[sat_mask] = np.nan

            poincare_bg_mask = pol.get_poincare_bg_mask()
            if poincare_bg_mask is None:
                poincare_bg_mask = np.zeros_like(bg_mask)

            from polarimetro import stokes as polstokes
            wav_intensity = polstokes.get_wav_intensity_cache()

            stokes_data[ch] = {
                'S0': S0, 'S1': S1, 'S2': S2, 'S3': S3,
                'bg_mask': bg_mask,
                'poincare_bg_mask': poincare_bg_mask.copy(),
                'sat_mask': sat_mask,
                'DoLP': DoLP, 'AoLP': AoLP,
                'delta': delta_deg, 'theta': theta_deg,
                'wav_intensity': (wav_intensity.copy()
                                  if wav_intensity is not None else None),
                'channel_idx': ch_idx,
            }

            valid = bg_mask & np.isfinite(delta_deg)
            if valid.any():
                p5, p50, p95 = np.percentile(delta_deg[valid], [5, 50, 95])
                print(f"  {ch}: S0 shape={S0.shape}  "
                      f"delta_bg pctl [5,50,95]=[{p5:.1f}, {p50:.1f}, {p95:.1f}] deg")

        save_dict = {
            'downsample_factor': np.int32(DOWNSAMPLE_FACTOR),
            'dataset': np.array(DATASET),
        }
        for ch, d in stokes_data.items():
            for k, v in d.items():
                if v is None or k == 'channel_idx':
                    continue
                save_dict[f'{ch}_{k}'] = v
        np.savez_compressed(cache_path, **save_dict)
        print(f"\nCache saved: {cache_path}")

## AB — 9 mappe (display + autosave PDF/HTML)

Per ogni canale attivo, genera in stile pubblicazione le 9 mappe `S0, S1, S2, S3, DoLP, AoLP, δ, θ, mask`. `plt.show()` sempre; `plt.savefig()` solo se `SAVE_PLOTS=True` in `OUTPUT_DIR/<DATASET>/<CH>_<param>.pdf`. HTML plotly interattivi (parametri non-mask) in `OUTPUT_DIR/<DATASET>/interactive/<CH>_<param>.html`.

In [ ]:
if 'AB' in active_analyses and stokes_data:
    from matplotlib.colors import LinearSegmentedColormap
    from matplotlib.patches import Patch
    try:
        import plotly.graph_objects as go
        _PLOTLY_OK = True
    except ImportError:
        _PLOTLY_OK = False

    AB_PARAM_CONFIG = {
        'S0':    {'titolo': 'Intensità totale $S_0$',                 'unita': 'conteggi (u.a.)', 'cmap': None,       'vmin': None,    'vmax': None},
        'S1':    {'titolo': 'Parametro di Stokes $S_1$',              'unita': 'conteggi (u.a.)', 'cmap': 'bwr',      'vmin': 'sym99', 'vmax': 'sym99'},
        'S2':    {'titolo': 'Parametro di Stokes $S_2$',              'unita': 'conteggi (u.a.)', 'cmap': 'bwr',      'vmin': 'sym99', 'vmax': 'sym99'},
        'S3':    {'titolo': 'Parametro di Stokes $S_3$',              'unita': 'conteggi (u.a.)', 'cmap': 'bwr',      'vmin': 'sym99', 'vmax': 'sym99'},
        'DoLP':  {'titolo': 'Grado di polarizzazione lineare (DoLP)', 'unita': None,              'cmap': 'viridis',  'vmin': 0,       'vmax': 1},
        'AoLP':  {'titolo': "Angolo di polarizzazione lineare (AoLP)", 'unita': '°',              'cmap': 'twilight', 'vmin': -90,     'vmax': 90},
        'delta': {'titolo': r'Ritardo di fase $\delta$',              'unita': '°',              'cmap': 'twilight', 'vmin': 0,       'vmax': 360},
        'theta': {'titolo': r'Asse veloce $\theta$',                  'unita': '°',              'cmap': 'twilight', 'vmin': -90,     'vmax': 90},
        'mask':  {'titolo': 'Maschera di sfondo',                     'unita': None,              'cmap': 'gray',     'vmin': 0,       'vmax': 1},
    }
    _S0_CMAPS = {
        0: LinearSegmentedColormap.from_list('nero_rosso', ['black', 'red']),
        1: LinearSegmentedColormap.from_list('nero_verde', ['black', 'green']),
        2: LinearSegmentedColormap.from_list('nero_blu',   ['black', 'blue']),
    }

    def _resolve_limits(data, vmin_spec, vmax_spec):
        if vmin_spec == 'sym99':
            bound = float(np.nanpercentile(np.abs(data), 99))
            return -bound, bound
        return vmin_spec, vmax_spec

    def _mpl_cmap_to_plotly(cmap, n=64):
        cmap_obj = plt.get_cmap(cmap) if isinstance(cmap, str) else cmap
        scale = []
        for s in np.linspace(0.0, 1.0, n):
            r, g, b, _ = cmap_obj(float(s))
            scale.append([float(s), f"rgb({int(255*r)},{int(255*g)},{int(255*b)})"])
        return scale

    def _save_interactive_html(data, param, ch_idx, out_path):
        if not _PLOTLY_OK:
            return False
        cfg = AB_PARAM_CONFIG[param]
        cmap = _S0_CMAPS[ch_idx] if cfg['cmap'] is None else cfg['cmap']
        vmin, vmax = _resolve_limits(data, cfg['vmin'], cfg['vmax'])
        H, W = data.shape
        fig = go.Figure(data=go.Heatmap(
            z=data,
            colorscale=_mpl_cmap_to_plotly(cmap),
            zmin=vmin, zmax=vmax,
            hovertemplate='x: %{x}<br>y: %{y}<br>valore: %{z:.4g}<extra></extra>',
            colorbar=dict(title=cfg['unita'] or ''),
        ))
        fig.update_layout(
            title=cfg['titolo'],
            xaxis=dict(scaleanchor='y', constrain='domain'),
            yaxis=dict(autorange='reversed'),
            width=min(1000, 80 + W),
            height=min(900, 80 + H),
            margin=dict(l=40, r=40, t=60, b=40),
        )
        try:
            fig.write_html(out_path, include_plotlyjs='cdn', full_html=True)
            return True
        except Exception as e:
            print(f"  (avviso: HTML plotly non scritto per {out_path}: {e})")
            return False

    def _make_figure(param, data, ch_idx):
        cfg = AB_PARAM_CONFIG[param]
        if param == 'mask' and data.ndim == 3:
            H, W, _ = data.shape
            aspect = H / W
            fig_w = 3.35
            fig, ax = plt.subplots(figsize=(fig_w + 0.2, fig_w * aspect))
            ax.imshow(data, aspect='equal')
            ax.set_title(cfg['titolo'], pad=6)
            ax.axis('off')
            handles = [
                Patch(facecolor='#bfbfbf', edgecolor='none', label='entrambe (S0)'),
                Patch(facecolor='#0040ff', edgecolor='none', label='XOR (wav debug)'),
                Patch(facecolor='#ff0000', edgecolor='none', label='nessuna (sample)'),
            ]
            ax.legend(handles=handles, loc='lower right', fontsize=6,
                      framealpha=0.85, handlelength=1.2,
                      borderpad=0.3, labelspacing=0.25)
            return fig
        cmap = _S0_CMAPS[ch_idx] if cfg['cmap'] is None else cfg['cmap']
        vmin, vmax = _resolve_limits(data, cfg['vmin'], cfg['vmax'])
        H, W = data.shape
        aspect = H / W
        fig_w = 3.35
        fig, ax = plt.subplots(figsize=(fig_w + 0.7, fig_w * aspect))
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
        ax.set_title(cfg['titolo'], pad=6)
        ax.axis('off')
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        if cfg['unita']:
            cbar.set_label(cfg['unita'])
        return fig

    out_dir = os.path.join(OUTPUT_DIR, DATASET)
    interactive_dir = os.path.join(out_dir, 'interactive')
    if SAVE_PLOTS:
        os.makedirs(out_dir, exist_ok=True)
        os.makedirs(interactive_dir, exist_ok=True)

    AB_PARAMS = ['S0', 'S1', 'S2', 'S3', 'DoLP', 'AoLP', 'delta', 'theta', 'mask']

    for ch in ACTIVE_CHANNELS:
        d = stokes_data[ch]
        ch_idx = d['channel_idx']
        wav_mean = d['wav_intensity'] / 2.0 if d['wav_intensity'] is not None else None
        mask_rgb = polplot.mask_overlay_rgb(
            d['S0'], d['bg_mask'], d['poincare_bg_mask'], wav_mean=wav_mean)
        data_map = {
            'S0': d['S0'], 'S1': d['S1'], 'S2': d['S2'], 'S3': d['S3'],
            'DoLP': d['DoLP'], 'AoLP': d['AoLP'],
            'delta': d['delta'], 'theta': d['theta'],
            'mask': mask_rgb,
        }
        for param in AB_PARAMS:
            data = data_map[param]
            if data is None:
                continue
            fig = _make_figure(param, data, ch_idx)
            pdf_path = os.path.join(out_dir, f"{ch}_{param}.pdf") if SAVE_PLOTS else None
            polplot.save_and_show(fig, pdf_path, show=True, fmt='pdf')
            plt.close(fig)
            if SAVE_PLOTS and param != 'mask':
                html_path = os.path.join(interactive_dir, f"{ch}_{param}.html")
                _save_interactive_html(data, param, ch_idx, html_path)

## C-aolp — UMAP scatter + AoLP map + AoLP histogram

Embedding UMAP sparso a risoluzione nativa, colorato per AoLP (cmap viridis, autoscala 1-99 pct). Default per zucchero.

In [ ]:
if 'C-aolp' in active_analyses:
    import time as _time

    UMAP_SAMPLE_N = polumap.UMAP_RANDOM_SAMPLE_N  # 10000
    _CHANNEL_SEED_OFFSET = {'R': 0, 'G': 1, 'B': 2}
    HIST_BINS_UMAP = 180
    CACHE_DIR_UMAP = './outputs'
    os.makedirs(CACHE_DIR_UMAP, exist_ok=True)

    def compute_or_load_umap_cache(ch, sd, n_target=UMAP_SAMPLE_N):
        cache_path = os.path.join(CACHE_DIR_UMAP,
                                  f"umap_{DATASET}_{ch}_cache.npz")
        if os.path.exists(cache_path):
            with np.load(cache_path) as d:
                ok = ('delta_deg' in d.files and 'axis_conf_min' in d.files
                      and 'sample_n' in d.files
                      and float(d['axis_conf_min']) == polumap.UMAP_AXIS_CONFIDENCE_MIN
                      and int(d['sample_n']) == n_target)
                if ok:
                    print(f"[cache] carico {cache_path}")
                    return (d['embedding'].copy(),
                            d['aolp_deg'].copy(),
                            d['delta_deg'].copy(),
                            d['valid_indices'].copy(),
                            tuple(int(x) for x in d['S0_shape']))
            print(f"[cache] {cache_path} obsoleto, ricomputo")
        S0 = sd['S0']; S1 = sd['S1']; S2 = sd['S2']; S3 = sd['S3']
        DoLP = sd['DoLP']; AoLP = sd['AoLP']
        delta = sd['delta']; theta = sd['theta']
        bg_mask = sd['bg_mask']; sat_mask = sd.get('sat_mask')
        base_valid = polumap.build_validity_mask(
            S0, DoLP, bg_mask, sat_mask=sat_mask, theta_deg=theta)
        valid_mask = polumap.random_sample_mask(S0.shape, base_valid,
                                                n_target=n_target,
                                                seed=polumap.RANDOM_STATE + _CHANNEL_SEED_OFFSET.get(ch, 0))
        features, valid_indices = polumap.build_feature_matrix(
            S0, S1, S2, S3, DoLP, valid_mask, feature_mode='no_delta')
        print(f"  pixel validi: {features.shape[0]}")
        if features.shape[0] < 100:
            print("  too few valid pixels, skip")
            return None
        t0 = _time.time()
        embedding = polumap.fit_umap(features)
        print(f"  UMAP fit: {_time.time()-t0:.1f}s")
        np.savez_compressed(
            cache_path, embedding=embedding, aolp_deg=AoLP,
            delta_deg=delta, valid_indices=valid_indices,
            S0_shape=np.array(S0.shape, dtype=np.int64),
            axis_conf_min=np.float32(polumap.UMAP_AXIS_CONFIDENCE_MIN),
            sample_n=np.int32(n_target))
        print(f"[cache] salvato {cache_path}")
        return embedding, AoLP, delta, valid_indices, S0.shape

    def export_umap_panels(spec, embedding, valid_indices, S0_shape,
                           dataset_label, channel_label,
                           hist_bins=HIST_BINS_UMAP):
        out = os.path.join(OUTPUT_DIR, dataset_label,
                           spec['export_subdir'], channel_label)
        os.makedirs(out, exist_ok=True)
        cmap = spec['cmap']
        vmin, vmax = spec['vmin'], spec['vmax']
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        H, W = S0_shape
        value_map = spec['value_map']
        value_valid = spec['value_valid']
        extend = spec['cbar_extend']

        aspect_img = H / W
        fig_w = 3.35
        fig_h = fig_w * aspect_img
        fig, ax = plt.subplots(figsize=(fig_w + 0.7, fig_h))
        im = ax.imshow(value_map, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
        ax.set_title(spec['map_title_short'], pad=6)
        ax.axis('off')
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, extend=extend)
        cb.set_label(spec['unit_label'])
        polplot.save_and_show(fig, os.path.join(out, spec['export_map_file']),
                              show=SAVE_PLOTS)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(3.35 + 0.7, 3.35))
        ax.scatter(embedding[:, 0], embedding[:, 1],
                   c=value_valid, cmap=cmap, vmin=vmin, vmax=vmax,
                   s=2.0, alpha=0.85)
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
        ax.set_title('UMAP embedding', pad=6)
        cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                          ax=ax, fraction=0.046, pad=0.03, extend=extend)
        cb.set_label(spec['unit_label'])
        polplot.save_and_show(fig, os.path.join(out, 'umap_scatter.pdf'),
                              show=SAVE_PLOTS)
        plt.close(fig)

        hmin, hmax = spec['hist_range']
        fig, ax = plt.subplots(figsize=(3.35, 2.2))
        finite = value_valid[np.isfinite(value_valid)]
        counts, edges = np.histogram(finite, bins=hist_bins, range=(hmin, hmax))
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        bar_colors = [cmap(norm(c)) for c in centers]
        ax.bar(centers, counts, width=widths, color=bar_colors,
               edgecolor='none', align='center')
        edge = spec['hist_edge_exclude']
        if edge > 0.0:
            inner = (centers > hmin + edge) & (centers < hmax - edge)
            ymax = (float(counts[inner].max())
                    if (inner.any() and counts[inner].size)
                    else (float(counts.max()) if counts.size else 1.0))
            ax.axvspan(hmin, hmin + edge, color='gray', alpha=0.08)
            ax.axvspan(hmax - edge, hmax, color='gray', alpha=0.08)
            ax.set_ylim(0, ymax * 1.25)
        else:
            ymax = float(np.percentile(counts, 99)) if counts.size else 1.0
            ax.set_ylim(0, ymax * 1.15)
        ax.set_xlim(hmin, hmax)
        ax.set_xlabel(spec['unit_label'])
        ax.set_ylabel('# pixel validi')
        ax.set_title(spec['hist_title'].replace('(c) ', ''), pad=6)
        ax.grid(True, axis='y', linestyle=':', alpha=0.4)
        polplot.save_and_show(fig, os.path.join(out, spec['export_hist_file']),
                              show=SAVE_PLOTS)
        plt.close(fig)
        return out

    UMAP_CACHE_BY_CHANNEL = {}
    for _ch in ACTIVE_CHANNELS:
        print(f"\n=== UMAP {DATASET} / {_ch} (color_by=aolp) ===")
        _sd = stokes_data.get(_ch)
        if _sd is None:
            print(f"  [{_ch}] skip: mancano stokes_data (eseguire Load+Stokes prima)")
            continue
        _result = compute_or_load_umap_cache(_ch, _sd)
        if _result is None:
            continue
        _embedding, _aolp, _delta, _valid_indices, _S0_shape = _result
        UMAP_CACHE_BY_CHANNEL[_ch] = _result
        _spec = polumap.color_spec('aolp', aolp_deg=_aolp,
                                   delta_deg=_delta,
                                   valid_indices=_valid_indices)
        _out = export_umap_panels(_spec, _embedding, _valid_indices, _S0_shape,
                                  DATASET, _ch)
        print(f"  [{_ch}/aolp] esportato: {_out}/")

## C-delta — UMAP scatter + δ map + δ histogram

Stesso fit di C-aolp (cache condivisa), colorato per retardance δ (cmap ciclica twilight 0-360, bande di esclusione ±20°). Default per strati/λ-quarti/λ-mezzi.

In [ ]:
if 'C-delta' in active_analyses:
    if 'UMAP_CACHE_BY_CHANNEL' not in globals():
        UMAP_CACHE_BY_CHANNEL = {}

    if 'compute_or_load_umap_cache' not in globals():
        import time as _time

        UMAP_SAMPLE_N = polumap.UMAP_RANDOM_SAMPLE_N  # 10000
    _CHANNEL_SEED_OFFSET = {'R': 0, 'G': 1, 'B': 2}
        HIST_BINS_UMAP = 180
        CACHE_DIR_UMAP = './outputs'
        os.makedirs(CACHE_DIR_UMAP, exist_ok=True)

        def _sparse_grid_mask(shape, stride):
            m = np.zeros(shape, dtype=bool)
            m[::stride, ::stride] = True
            return m

        def compute_or_load_umap_cache(ch, sd, n_target=UMAP_SAMPLE_N):
            cache_path = os.path.join(CACHE_DIR_UMAP,
                                      f"umap_{DATASET}_{ch}_cache.npz")
            if os.path.exists(cache_path):
                with np.load(cache_path) as d:
                    ok = ('delta_deg' in d.files and 'axis_conf_min' in d.files
                          and float(d['axis_conf_min']) == polumap.UMAP_AXIS_CONFIDENCE_MIN)
                    if ok:
                        print(f"[cache] carico {cache_path}")
                        return (d['embedding'].copy(),
                                d['aolp_deg'].copy(),
                                d['delta_deg'].copy(),
                                d['valid_indices'].copy(),
                                tuple(int(x) for x in d['S0_shape']))
                print(f"[cache] {cache_path} obsoleto, ricomputo")
            S0 = sd['S0']; S1 = sd['S1']; S2 = sd['S2']; S3 = sd['S3']
            DoLP = sd['DoLP']; AoLP = sd['AoLP']
            delta = sd['delta']; theta = sd['theta']
            bg_mask = sd['bg_mask']; sat_mask = sd.get('sat_mask')
            base_valid = polumap.build_validity_mask(
                S0, DoLP, bg_mask, sat_mask=sat_mask, theta_deg=theta)
            valid_mask = polumap.random_sample_mask(S0.shape, base_valid,
                                                n_target=n_target,
                                                seed=polumap.RANDOM_STATE + _CHANNEL_SEED_OFFSET.get(ch, 0))
            features, valid_indices = polumap.build_feature_matrix(
                S0, S1, S2, S3, DoLP, valid_mask, feature_mode='no_delta')
            print(f"  pixel validi: {features.shape[0]}")
            if features.shape[0] < 100:
                print("  too few valid pixels, skip")
                return None
            t0 = _time.time()
            embedding = polumap.fit_umap(features)
            print(f"  UMAP fit: {_time.time()-t0:.1f}s")
            np.savez_compressed(
                cache_path, embedding=embedding, aolp_deg=AoLP,
                delta_deg=delta, valid_indices=valid_indices,
                S0_shape=np.array(S0.shape, dtype=np.int64),
                axis_conf_min=np.float32(polumap.UMAP_AXIS_CONFIDENCE_MIN),
            sample_n=np.int32(n_target))
            print(f"[cache] salvato {cache_path}")
            return embedding, AoLP, delta, valid_indices, S0.shape

        def export_umap_panels(spec, embedding, valid_indices, S0_shape,
                               dataset_label, channel_label,
                               hist_bins=HIST_BINS_UMAP):
            out = os.path.join(OUTPUT_DIR, dataset_label,
                               spec['export_subdir'], channel_label)
            os.makedirs(out, exist_ok=True)
            cmap = spec['cmap']
            vmin, vmax = spec['vmin'], spec['vmax']
            norm = plt.Normalize(vmin=vmin, vmax=vmax)
            H, W = S0_shape
            value_map = spec['value_map']
            value_valid = spec['value_valid']
            extend = spec['cbar_extend']

            aspect_img = H / W
            fig_w = 3.35
            fig_h = fig_w * aspect_img
            fig, ax = plt.subplots(figsize=(fig_w + 0.7, fig_h))
            im = ax.imshow(value_map, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
            ax.set_title(spec['map_title_short'], pad=6)
            ax.axis('off')
            cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, extend=extend)
            cb.set_label(spec['unit_label'])
            polplot.save_and_show(fig, os.path.join(out, spec['export_map_file']),
                                  show=SAVE_PLOTS)
            plt.close(fig)

            fig, ax = plt.subplots(figsize=(3.35 + 0.7, 3.35))
            ax.scatter(embedding[:, 0], embedding[:, 1],
                       c=value_valid, cmap=cmap, vmin=vmin, vmax=vmax,
                       s=2.0, alpha=0.85)
            ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
            ax.set_title('UMAP embedding', pad=6)
            cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                              ax=ax, fraction=0.046, pad=0.03, extend=extend)
            cb.set_label(spec['unit_label'])
            polplot.save_and_show(fig, os.path.join(out, 'umap_scatter.pdf'),
                                  show=SAVE_PLOTS)
            plt.close(fig)

            hmin, hmax = spec['hist_range']
            fig, ax = plt.subplots(figsize=(3.35, 2.2))
            finite = value_valid[np.isfinite(value_valid)]
            counts, edges = np.histogram(finite, bins=hist_bins, range=(hmin, hmax))
            centers = 0.5 * (edges[:-1] + edges[1:])
            widths = np.diff(edges)
            bar_colors = [cmap(norm(c)) for c in centers]
            ax.bar(centers, counts, width=widths, color=bar_colors,
                   edgecolor='none', align='center')
            edge = spec['hist_edge_exclude']
            if edge > 0.0:
                inner = (centers > hmin + edge) & (centers < hmax - edge)
                ymax = (float(counts[inner].max())
                        if (inner.any() and counts[inner].size)
                        else (float(counts.max()) if counts.size else 1.0))
                ax.axvspan(hmin, hmin + edge, color='gray', alpha=0.08)
                ax.axvspan(hmax - edge, hmax, color='gray', alpha=0.08)
                ax.set_ylim(0, ymax * 1.25)
            else:
                ymax = float(np.percentile(counts, 99)) if counts.size else 1.0
                ax.set_ylim(0, ymax * 1.15)
            ax.set_xlim(hmin, hmax)
            ax.set_xlabel(spec['unit_label'])
            ax.set_ylabel('# pixel validi')
            ax.set_title(spec['hist_title'].replace('(c) ', ''), pad=6)
            ax.grid(True, axis='y', linestyle=':', alpha=0.4)
            polplot.save_and_show(fig, os.path.join(out, spec['export_hist_file']),
                                  show=SAVE_PLOTS)
            plt.close(fig)
            return out

    for _ch in ACTIVE_CHANNELS:
        print(f"\n=== UMAP {DATASET} / {_ch} (color_by=delta) ===")
        _result = UMAP_CACHE_BY_CHANNEL.get(_ch)
        if _result is None:
            _sd = stokes_data.get(_ch)
            if _sd is None:
                print(f"  [{_ch}] skip: mancano stokes_data")
                continue
            _result = compute_or_load_umap_cache(_ch, _sd)
            if _result is None:
                continue
            UMAP_CACHE_BY_CHANNEL[_ch] = _result
        _embedding, _aolp, _delta, _valid_indices, _S0_shape = _result
        _spec = polumap.color_spec('delta', aolp_deg=_aolp,
                                   delta_deg=_delta,
                                   valid_indices=_valid_indices)
        _out = export_umap_panels(_spec, _embedding, _valid_indices, _S0_shape,
                                  DATASET, _ch)
        print(f"  [{_ch}/delta] esportato: {_out}/")

## C-debug — Diagnostico campionamento UMAP (random sample)

Mostra dove sono caduti i punti del campionamento random nello spazio immagine. Attivo se almeno una fra `C-aolp` e `C-delta` è in `active_analyses`.

In [ ]:
if ('C-aolp' in active_analyses or 'C-delta' in active_analyses) and stokes_data:
    _diag_chs = [ch for ch in ACTIVE_CHANNELS if ch in UMAP_CACHE_BY_CHANNEL]
    if _diag_chs:
        _ncols = len(_diag_chs)
        _fig, _axs = plt.subplots(1, _ncols, figsize=(3.5*_ncols, 3.5))
        if _ncols == 1:
            _axs = [_axs]
        for _ax, _ch in zip(_axs, _diag_chs):
            _embedding, _aolp, _delta, _valid_indices, _S0_shape = UMAP_CACHE_BY_CHANNEL[_ch]
            _sample_mask = np.zeros(_S0_shape, dtype=bool)
            _flat = np.array(_valid_indices, dtype=np.int64)
            _ys, _xs = np.unravel_index(_flat, _S0_shape)
            _sample_mask[_ys, _xs] = True
            polumap.plot_sample_diagnostic(_ax, stokes_data[_ch]['S0'], _sample_mask,
                                            bg_mask=stokes_data[_ch]['bg_mask'],
                                            title=f'{DATASET} / {_ch} — N={len(_flat)}')
        plt.tight_layout()
        if SAVE_PLOTS:
            _out = os.path.join(OUTPUT_DIR, DATASET, 'umap_sample_diagnostic.pdf')
            os.makedirs(os.path.dirname(_out), exist_ok=True)
            _fig.savefig(_out, dpi=300, bbox_inches='tight')
            print(f'debug sample plot: {_out}')
        plt.show()


## E-ext — Istogrammi δ pubblicabili (strati)

PDF + HTML interattivo con barre colorate twilight. Solo `strati_v2`.

In [ ]:
if 'E-ext' in active_analyses:
    import scipy.ndimage as _ndimage
    try:
        import plotly.graph_objects as _go
        _PLOTLY_OK_E = True
    except ImportError:
        _PLOTLY_OK_E = False

    HIST_BINS_E = 180
    HIST_EDGE_EXCLUDE_E = 20.0
    SLICE_ANCHOR_NATIVE_XY = (767, 422)
    SLICE_ANGLE_DEG_E = 141.0
    SLICE_HALF_WIDTH_NATIVE_PX_E = 200.0
    SLICE_STEP_NATIVE_PX_E = 5.0
    AUTO_CROP_EDGE_MARGIN_DEG_E = 15.0
    AUTO_CROP_IGNORE_NATIVE_PX_E = 1500.0
    PLATEAU_SMOOTH_SIGMA_E = 1.5
    PLATEAU_GRAD_THRESH_DEG_E = 1.0
    PLATEAU_MIN_LENGTH_NATIVE_PX_E = 30.0
    PLATEAU_LABELS_E = ['1L', '2L', '3L', '4L', '5', '4R', '3R', '2R', '1R']

    def _build_slice_grid_E(shape, anchor_native, angle_deg, ds,
                            half_width_native, step_native):
        H, W = shape
        cx_ds = anchor_native[0] / ds
        cy_ds = anchor_native[1] / ds
        half_ds = half_width_native / ds
        step_ds = step_native / ds
        theta = np.deg2rad(angle_deg)
        dx, dy = np.cos(theta), -np.sin(theta)

        def t_to_edge(c, d, lim):
            if d > 1e-12: return (lim - 1 - c) / d
            if d < -1e-12: return -c / d
            return np.inf
        t_pos = min(t_to_edge(cx_ds, dx, W), t_to_edge(cy_ds, dy, H))
        t_neg = min(t_to_edge(cx_ds, -dx, W), t_to_edge(cy_ds, -dy, H))
        t_vals = np.arange(-t_neg, t_pos + step_ds, step_ds)
        nx, ny = -dy, dx
        n_perp = max(0, int(np.floor(half_ds)))
        offsets = np.arange(-n_perp, n_perp + 1)
        T, U = np.meshgrid(t_vals, offsets, indexing='ij')
        X = cx_ds + T * dx + U * nx
        Y = cy_ds + T * dy + U * ny
        return t_vals, X, Y, (cx_ds, cy_ds)

    def _sample_thick_slice_E(delta_map, X, Y):
        valid_in = np.isfinite(delta_map)
        delta_filled = np.where(valid_in, delta_map, 0.0)
        rad = np.deg2rad(delta_filled)
        coords = np.stack([Y.ravel(), X.ravel()], axis=0)
        sin_s = _ndimage.map_coordinates(np.sin(rad), coords, order=1, mode='constant', cval=0.0)
        cos_s = _ndimage.map_coordinates(np.cos(rad), coords, order=1, mode='constant', cval=0.0)
        valid_s = _ndimage.map_coordinates(valid_in.astype(np.float32), coords, order=1, mode='constant', cval=0.0)
        sin_s = sin_s.reshape(X.shape); cos_s = cos_s.reshape(X.shape)
        w = (valid_s.reshape(X.shape) > 0.5).astype(np.float32)
        w_sum = w.sum(axis=1); w_safe = np.maximum(w_sum, 1.0)
        sin_m = (sin_s * w).sum(axis=1) / w_safe
        cos_m = (cos_s * w).sum(axis=1) / w_safe
        delta_mean = np.degrees(np.arctan2(sin_m, cos_m)) % 360.0
        delta_mean = np.where(w_sum > 0.5, delta_mean, np.nan)
        return delta_mean

    def _find_sample_crop_E(delta_slice, t_vals, ds, edge_margin, ignore_native):
        n = len(delta_slice)
        if n == 0: return 0, 0
        arc_native = t_vals * ds
        arc_min = float(arc_native[0]) + ignore_native
        arc_max = float(arc_native[-1]) - ignore_native
        in_range = (arc_native >= arc_min) & (arc_native <= arc_max)
        in_win = (np.isfinite(delta_slice)
                  & (delta_slice >= edge_margin)
                  & (delta_slice <= 360.0 - edge_margin)
                  & in_range)
        idx = np.where(in_win)[0]
        if idx.size == 0:
            r = np.where(in_range)[0]
            return (int(r[0]), int(r[-1])) if r.size else (0, n - 1)
        return int(idx[0]), int(idx[-1])

    def _detect_plateaus_E(delta_slice, t_vals, ds, crop_idx,
                           sigma, grad_thresh, min_len_native, labels):
        s, e = crop_idx
        if e <= s: return []
        seg = delta_slice[s:e + 1]
        valid = np.isfinite(seg)
        if valid.sum() < 3: return []
        seg_for_smooth = np.where(valid, seg, np.nanmean(seg) if valid.any() else 0.0)
        smooth = _ndimage.gaussian_filter1d(seg_for_smooth, sigma=sigma, mode='nearest')
        grad = np.abs(np.gradient(smooth))
        flat = (grad < grad_thresh) & valid
        step_native = (t_vals[1] - t_vals[0]) * ds if len(t_vals) > 1 else 1.0
        min_run = max(1, int(np.ceil(min_len_native / max(step_native, 1e-6))))
        padded = np.concatenate(([False], flat, [False]))
        diffs = np.diff(padded.astype(np.int8))
        starts = np.where(diffs == 1)[0]
        ends_excl = np.where(diffs == -1)[0]
        runs = [(int(a), int(b - 1)) for a, b in zip(starts, ends_excl)
                if (b - a) >= min_run]
        if not runs: return []
        n_exp = len(labels)
        if len(runs) > n_exp:
            runs = sorted(runs, key=lambda r: -(r[1] - r[0]))[:n_exp]
        runs = sorted(runs, key=lambda r: r[0])
        arc_native = t_vals * ds
        seg_arc = arc_native[s:e + 1]
        plateaus = []
        seg_filled = np.where(valid, seg, np.nan)
        for i, (a, b) in enumerate(runs):
            lbl = labels[i] if len(runs) == n_exp else f"P{i+1}"
            block = seg_filled[a:b + 1]
            block = block[np.isfinite(block)]
            if block.size == 0: continue
            rad = np.deg2rad(block)
            med = np.degrees(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean())) % 360.0
            plateaus.append({
                'label': lbl,
                'arc_mid': 0.5 * (float(seg_arc[a]) + float(seg_arc[b])),
                'delta_med': float(med),
            })
        return plateaus

    out_dir_E = os.path.join(OUTPUT_DIR, DATASET)
    out_html_dir_E = os.path.join(out_dir_E, 'interactive')
    os.makedirs(out_dir_E, exist_ok=True)
    os.makedirs(out_html_dir_E, exist_ok=True)

    _cmap_E = plt.get_cmap('twilight')

    for _ch in ACTIVE_CHANNELS:
        _sd = stokes_data.get(_ch)
        if _sd is None:
            print(f"  [E-ext {_ch}] skip: mancano stokes_data")
            continue
        print(f"\n=== E-ext {DATASET} / {_ch} ===")
        _delta = _sd['delta']
        _bg = _sd['bg_mask']
        _DoLP = _sd['DoLP']
        _theta = _sd['theta']
        _sat = _sd.get('sat_mask')
        _valid = polumap.build_validity_mask(
            _sd['S0'], _DoLP, _bg, sat_mask=_sat, theta_deg=_theta)
        _delta_valid = _delta[_valid]
        _delta_valid = _delta_valid[np.isfinite(_delta_valid)]
        print(f"  pixel validi: {_delta_valid.size}")

        _t_vals, _X, _Y, _ = _build_slice_grid_E(
            _delta.shape, SLICE_ANCHOR_NATIVE_XY, SLICE_ANGLE_DEG_E,
            DOWNSAMPLE_FACTOR, SLICE_HALF_WIDTH_NATIVE_PX_E,
            SLICE_STEP_NATIVE_PX_E)
        _slice_prof = _sample_thick_slice_E(_delta, _X, _Y)
        _crop_idx = _find_sample_crop_E(_slice_prof, _t_vals, DOWNSAMPLE_FACTOR,
                                        AUTO_CROP_EDGE_MARGIN_DEG_E,
                                        AUTO_CROP_IGNORE_NATIVE_PX_E)
        _plateaus = _detect_plateaus_E(
            _slice_prof, _t_vals, DOWNSAMPLE_FACTOR, _crop_idx,
            PLATEAU_SMOOTH_SIGMA_E, PLATEAU_GRAD_THRESH_DEG_E,
            PLATEAU_MIN_LENGTH_NATIVE_PX_E, PLATEAU_LABELS_E)
        print(f"  plateau rilevati: {len(_plateaus)}")

        _counts, _edges = np.histogram(_delta_valid, bins=HIST_BINS_E, range=(0, 360))
        _centers = 0.5 * (_edges[:-1] + _edges[1:])
        _widths = np.diff(_edges)
        _bar_colors = [_cmap_E(c / 360.0) for c in _centers]

        _inner = (_centers > HIST_EDGE_EXCLUDE_E) & (_centers < 360 - HIST_EDGE_EXCLUDE_E)
        _ymax_inner = (float(_counts[_inner].max())
                       if _inner.any() and _counts[_inner].size
                       else float(_counts.max()) if _counts.size else 1.0)

        fig, ax = plt.subplots(figsize=(4.5, 2.8))
        ax.bar(_centers, _counts, width=_widths, color=_bar_colors,
               edgecolor='none', align='center')
        for p in _plateaus:
            ax.axvline(p['delta_med'], color='black', linestyle='--',
                       linewidth=0.7, alpha=0.7, zorder=4)
            ax.annotate(p['label'], xy=(p['delta_med'], _ymax_inner * 1.05),
                        ha='center', va='bottom', fontsize=6,
                        color='black', rotation=0)
        ax.set_xlim(0, 360)
        ax.set_ylim(0, _ymax_inner * 1.30)
        ax.set_xticks(np.arange(0, 361, 60))
        ax.set_xlabel(r"$\delta$ (°)")
        ax.set_ylabel('# pixel validi')
        ax.set_title(fr"Istogramma $\delta$ - {DATASET} canale {_ch}", pad=6)
        ax.axvspan(0, HIST_EDGE_EXCLUDE_E, color='gray', alpha=0.08, linewidth=0)
        ax.axvspan(360 - HIST_EDGE_EXCLUDE_E, 360, color='gray', alpha=0.08, linewidth=0)
        ax.grid(True, axis='y', linestyle=':', alpha=0.4)
        _pdf_path = os.path.join(out_dir_E, f"{_ch}_hist_delta.pdf")
        polplot.save_and_show(fig, _pdf_path, show=SAVE_PLOTS)
        plt.close(fig)

        if _PLOTLY_OK_E:
            _bar_rgb = [f"rgb({int(255*r)},{int(255*g)},{int(255*b)})"
                        for r, g, b, _ in (_cmap_E(c / 360.0) for c in _centers)]
            _hover = (r"delta: %{x:.1f}°<br># pixel: %{y}<extra></extra>")
            figh = _go.Figure(data=_go.Bar(
                x=_centers, y=_counts, width=_widths,
                marker=dict(color=_bar_rgb, line=dict(width=0)),
                hovertemplate=_hover, name=f"{DATASET} {_ch}",
            ))
            figh.add_vrect(x0=0, x1=HIST_EDGE_EXCLUDE_E,
                           fillcolor='lightgray', opacity=0.25, line_width=0)
            figh.add_vrect(x0=360 - HIST_EDGE_EXCLUDE_E, x1=360,
                           fillcolor='lightgray', opacity=0.25, line_width=0)
            for p in _plateaus:
                figh.add_vline(x=p['delta_med'], line_dash='dash',
                               line_color='black', line_width=1,
                               annotation_text=p['label'],
                               annotation_position='top',
                               annotation_font_size=10)
            figh.update_layout(
                title=f"Istogramma δ - {DATASET} canale {_ch}",
                xaxis=dict(title='δ (°)', range=[0, 360],
                           tickmode='array', tickvals=list(range(0, 361, 30))),
                yaxis=dict(title='# pixel validi', range=[0, _ymax_inner * 1.30]),
                bargap=0, width=900, height=500,
                margin=dict(l=60, r=30, t=60, b=50),
                template='simple_white',
            )
            _html_path = os.path.join(out_html_dir_E, f"{_ch}_hist_delta.html")
            figh.write_html(_html_path, include_plotlyjs='cdn', full_html=True)
            print(f"  HTML salvato: {_html_path}")

## F — Slice diagonale δ multistrato (publication-style)

Tre pannelli (mappa δ con banda evidenziata | profilo 1D etichettato 1L-5-1R | fit through-origin con unwrap per-side). Solo `strati_v2`.

In [ ]:
if 'F' in active_analyses:
    import scipy.ndimage as _ndimage_F
    from matplotlib.lines import Line2D as _Line2D_F
    try:
        import plotly.graph_objects as _go_F
        _PLOTLY_OK_F = True
    except ImportError:
        _PLOTLY_OK_F = False

    SLICE_ANCHOR_F = (767, 422)
    SLICE_ANGLE_F = 141.0
    SLICE_HALF_WIDTH_NATIVE_F = 200.0
    SLICE_STEP_NATIVE_F = 5.0
    AUTO_CROP_MARGIN_F = 15.0
    AUTO_CROP_IGNORE_NATIVE_F = 1500.0
    PLATEAU_SIGMA_F = 1.5
    PLATEAU_GRAD_F = 1.0
    PLATEAU_MIN_LEN_NATIVE_F = 30.0
    PLATEAU_LABELS_F = ['1L', '2L', '3L', '4L', '5', '4R', '3R', '2R', '1R']

    def _build_slice_grid_F(shape, anchor_native, angle_deg, ds, hw_native, step_native):
        H, W = shape
        cx_ds = anchor_native[0] / ds; cy_ds = anchor_native[1] / ds
        half_ds = hw_native / ds; step_ds = step_native / ds
        theta = np.deg2rad(angle_deg)
        dx, dy = np.cos(theta), -np.sin(theta)

        def t_to_edge(c, d, lim):
            if d > 1e-12: return (lim - 1 - c) / d
            if d < -1e-12: return -c / d
            return np.inf
        t_pos = min(t_to_edge(cx_ds, dx, W), t_to_edge(cy_ds, dy, H))
        t_neg = min(t_to_edge(cx_ds, -dx, W), t_to_edge(cy_ds, -dy, H))
        t_vals = np.arange(-t_neg, t_pos + step_ds, step_ds)
        nx, ny = -dy, dx
        n_perp = max(0, int(np.floor(half_ds)))
        offsets = np.arange(-n_perp, n_perp + 1)
        T, U = np.meshgrid(t_vals, offsets, indexing='ij')
        X = cx_ds + T * dx + U * nx
        Y = cy_ds + T * dy + U * ny
        return t_vals, X, Y, (cx_ds, cy_ds)

    def _sample_thick_F(delta_map, X, Y):
        valid_in = np.isfinite(delta_map)
        delta_filled = np.where(valid_in, delta_map, 0.0)
        rad = np.deg2rad(delta_filled)
        coords = np.stack([Y.ravel(), X.ravel()], axis=0)
        sin_s = _ndimage_F.map_coordinates(np.sin(rad), coords, order=1, mode='constant', cval=0.0)
        cos_s = _ndimage_F.map_coordinates(np.cos(rad), coords, order=1, mode='constant', cval=0.0)
        valid_s = _ndimage_F.map_coordinates(valid_in.astype(np.float32), coords, order=1, mode='constant', cval=0.0)
        sin_s = sin_s.reshape(X.shape); cos_s = cos_s.reshape(X.shape)
        w = (valid_s.reshape(X.shape) > 0.5).astype(np.float32)
        w_sum = w.sum(axis=1); w_safe = np.maximum(w_sum, 1.0)
        sin_m = (sin_s * w).sum(axis=1) / w_safe
        cos_m = (cos_s * w).sum(axis=1) / w_safe
        delta_mean = np.degrees(np.arctan2(sin_m, cos_m)) % 360.0
        delta_mean = np.where(w_sum > 0.5, delta_mean, np.nan)
        R = np.sqrt(sin_m ** 2 + cos_m ** 2)
        circ_std = np.degrees(np.sqrt(np.maximum(-2.0 * np.log(np.clip(R, 1e-9, 1.0)), 0.0)))
        circ_std = np.where(w_sum > 0.5, circ_std, np.nan)
        return delta_mean, circ_std

    def _find_crop_F(profile, t_vals, ds, margin, ignore_native):
        n = len(profile)
        if n == 0: return 0, 0
        arc = t_vals * ds
        arc_min = float(arc[0]) + ignore_native
        arc_max = float(arc[-1]) - ignore_native
        in_range = (arc >= arc_min) & (arc <= arc_max)
        in_win = (np.isfinite(profile)
                  & (profile >= margin)
                  & (profile <= 360.0 - margin)
                  & in_range)
        idx = np.where(in_win)[0]
        if idx.size == 0:
            r = np.where(in_range)[0]
            return (int(r[0]), int(r[-1])) if r.size else (0, n - 1)
        return int(idx[0]), int(idx[-1])

    def _detect_plateaus_F(profile, t_vals, ds, crop_idx, sigma, grad_th,
                           min_len_native, labels):
        s, e = crop_idx
        if e <= s: return []
        seg = profile[s:e + 1]
        valid = np.isfinite(seg)
        if valid.sum() < 3: return []
        seg_for_smooth = np.where(valid, seg, np.nanmean(seg) if valid.any() else 0.0)
        smooth = _ndimage_F.gaussian_filter1d(seg_for_smooth, sigma=sigma, mode='nearest')
        grad = np.abs(np.gradient(smooth))
        flat = (grad < grad_th) & valid
        step_native = (t_vals[1] - t_vals[0]) * ds if len(t_vals) > 1 else 1.0
        min_run = max(1, int(np.ceil(min_len_native / max(step_native, 1e-6))))
        padded = np.concatenate(([False], flat, [False]))
        diffs = np.diff(padded.astype(np.int8))
        starts = np.where(diffs == 1)[0]
        ends_excl = np.where(diffs == -1)[0]
        runs = [(int(a), int(b - 1)) for a, b in zip(starts, ends_excl)
                if (b - a) >= min_run]
        if not runs: return []
        n_exp = len(labels)
        if len(runs) > n_exp:
            runs = sorted(runs, key=lambda r: -(r[1] - r[0]))[:n_exp]
        runs = sorted(runs, key=lambda r: r[0])
        arc = t_vals * ds
        seg_arc = arc[s:e + 1]
        plateaus = []
        seg_filled = np.where(valid, seg, np.nan)
        for i, (a, b) in enumerate(runs):
            lbl = labels[i] if len(runs) == n_exp else f"P{i+1}"
            block = seg_filled[a:b + 1]
            block = block[np.isfinite(block)]
            if block.size == 0: continue
            rad = np.deg2rad(block)
            med = np.degrees(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean())) % 360.0
            plateaus.append({
                'label': lbl,
                'idx_start': s + a, 'idx_end': s + b,
                'arc_start': float(seg_arc[a]), 'arc_end': float(seg_arc[b]),
                'arc_mid': 0.5 * (float(seg_arc[a]) + float(seg_arc[b])),
                'delta_med': float(med), 'delta_std': float(np.std(block)),
                'n_samples': int(b - a + 1),
            })
        return plateaus

    def _unwrap_along_n_F(items):
        if not items: return []
        out = [(items[0][0], items[0][1], items[0][2])]
        offset = 0.0; prev = items[0][2]
        for n, lbl, y in items[1:]:
            y_adj = y + offset
            while y_adj < prev:
                offset += 360.0; y_adj += 360.0
            out.append((n, lbl, y_adj)); prev = y_adj
        return out

    def _fit_layers_F(plateaus):
        L, R, C = [], [], []
        for p in plateaus:
            lbl = p['label']; y = float(p['delta_med'])
            if lbl.endswith('L'):
                try: n = int(lbl[:-1])
                except ValueError: continue
                L.append((n, lbl, y))
            elif lbl.endswith('R'):
                try: n = int(lbl[:-1])
                except ValueError: continue
                R.append((n, lbl, y))
            else:
                try: n = int(lbl)
                except ValueError: continue
                C.append((n, lbl, y))
        if not (L or R or C): return None
        L_seq = sorted(L + C, key=lambda t: t[0])
        R_seq = sorted(R + C, key=lambda t: t[0])
        L_unw = _unwrap_along_n_F(L_seq); R_unw = _unwrap_along_n_F(R_seq)
        unwrapped = {}
        for _, lbl, y in L_unw: unwrapped.setdefault(lbl, []).append(y)
        for _, lbl, y in R_unw: unwrapped.setdefault(lbl, []).append(y)
        pts = []
        for p in plateaus:
            lbl = p['label']
            if lbl not in unwrapped: continue
            try: n = int(lbl.rstrip('LR'))
            except ValueError: continue
            y_unw = float(np.mean(unwrapped[lbl]))
            pts.append((n, y_unw, lbl, float(p['delta_med'])))
        if not pts: return None
        x = np.array([q[0] for q in pts], dtype=float)
        y = np.array([q[1] for q in pts], dtype=float)
        y_raw = np.array([q[3] for q in pts], dtype=float)
        denom = float((x * x).sum())
        if denom <= 0: return None
        m = float((x * y).sum() / denom)
        y_pred = m * x
        ss_res = float(((y - y_pred) ** 2).sum())
        ss_tot = float(((y - y.mean()) ** 2).sum())
        r2 = (1.0 - ss_res / ss_tot) if ss_tot > 0 else float('nan')
        rms = float(np.sqrt(ss_res / max(len(y), 1)))
        n_unw = int(np.sum(np.abs(y - y_raw) > 1e-6))
        return {'slope': m, 'r2': r2, 'rms': rms, 'x': x, 'y': y,
                'y_raw': y_raw, 'labels': [q[2] for q in pts],
                'n_unwrapped': n_unw}

    _out_F = os.path.join(OUTPUT_DIR, DATASET)
    _out_html_F = os.path.join(_out_F, 'interactive')
    os.makedirs(_out_F, exist_ok=True)
    os.makedirs(_out_html_F, exist_ok=True)

    for _ch in ACTIVE_CHANNELS:
        _sd = stokes_data.get(_ch)
        if _sd is None:
            print(f"  [F {_ch}] skip: mancano stokes_data"); continue
        print(f"\n=== F slice {DATASET} / {_ch} ===")
        _delta = _sd['delta']
        _ch_idx = _sd['channel_idx']
        _wavelength = polcfg.get_channel_wavelength(WAVELENGTHS_CSV, _ch_idx)

        _t_vals, _X, _Y, _center = _build_slice_grid_F(
            _delta.shape, SLICE_ANCHOR_F, SLICE_ANGLE_F,
            DOWNSAMPLE_FACTOR, SLICE_HALF_WIDTH_NATIVE_F, SLICE_STEP_NATIVE_F)
        _profile, _circ_std = _sample_thick_F(_delta, _X, _Y)
        _crop_idx = _find_crop_F(_profile, _t_vals, DOWNSAMPLE_FACTOR,
                                 AUTO_CROP_MARGIN_F, AUTO_CROP_IGNORE_NATIVE_F)
        _plateaus = _detect_plateaus_F(
            _profile, _t_vals, DOWNSAMPLE_FACTOR, _crop_idx,
            PLATEAU_SIGMA_F, PLATEAU_GRAD_F, PLATEAU_MIN_LEN_NATIVE_F,
            PLATEAU_LABELS_F)
        _fit = _fit_layers_F(_plateaus) if _plateaus else None
        print(f"  plateau rilevati: {len(_plateaus)}, fit: {'OK' if _fit else 'none'}")

        if _fit is not None:
            fig, axes = plt.subplots(
                1, 3, figsize=(11.0, 3.6),
                gridspec_kw={'width_ratios': [1.0, 1.4, 0.7]})
        else:
            fig, axes = plt.subplots(
                1, 2, figsize=(8.5, 3.6),
                gridspec_kw={'width_ratios': [1.0, 1.3]})

        ax_map = axes[0]
        im = ax_map.imshow(_delta, cmap='twilight', vmin=0, vmax=360, origin='upper')
        cb = fig.colorbar(im, ax=ax_map, fraction=0.046, pad=0.03)
        cb.set_label(r'$\delta$ (°)')
        mid_col = _X.shape[1] // 2
        s_c, e_c = _crop_idx
        sl = slice(s_c, e_c + 1)
        ax_map.plot(_X[:, mid_col], _Y[:, mid_col], '-', color='gray', lw=0.5, alpha=0.5)
        ax_map.plot(_X[:, 0], _Y[:, 0], 'w-', lw=0.4, alpha=0.4)
        ax_map.plot(_X[:, -1], _Y[:, -1], 'w-', lw=0.4, alpha=0.4)
        ax_map.plot(_X[sl, mid_col], _Y[sl, mid_col], 'r-', lw=1.0)
        ax_map.plot(_X[sl, 0], _Y[sl, 0], 'y-', lw=0.6, alpha=0.9)
        ax_map.plot(_X[sl, -1], _Y[sl, -1], 'y-', lw=0.6, alpha=0.9)
        ax_map.plot(_center[0], _center[1], 'r+', mew=1.5, ms=8)
        ax_map.set_xlim(0, _delta.shape[1] - 1)
        ax_map.set_ylim(_delta.shape[0] - 1, 0)
        ax_map.set_xticks([]); ax_map.set_yticks([])
        ax_map.set_title(fr'Mappa di $\delta$ - canale {_ch} '
                         fr'($\lambda={_wavelength:.0f}$ nm)', pad=6)

        ax_prof = axes[1]
        arc_native = _t_vals * DOWNSAMPLE_FACTOR
        ax_prof.plot(arc_native[sl], _profile[sl], 'k-', lw=1.1,
                     label=r'$\delta$ medio (banda)')
        ax_prof.fill_between(arc_native[sl],
                             _profile[sl] - _circ_std[sl],
                             _profile[sl] + _circ_std[sl],
                             color='gray', alpha=0.25,
                             label=r'$\pm \sigma$ circolare')
        ax_prof.axhline(0, color='lightgray', lw=0.4)
        ax_prof.axhline(360, color='lightgray', lw=0.4)
        for p in _plateaus:
            ax_prof.plot([p['arc_start'], p['arc_end']],
                         [p['delta_med'], p['delta_med']],
                         color='tab:red', lw=2.0, solid_capstyle='butt', zorder=5)
            ax_prof.axvspan(p['arc_start'], p['arc_end'],
                            color='tab:red', alpha=0.05, zorder=0)
            ax_prof.annotate(
                f"{p['label']}\n{p['delta_med']:.1f}°",
                xy=(p['arc_mid'], p['delta_med']),
                xytext=(0, 10), textcoords='offset points',
                ha='center', va='bottom', fontsize=7, color='tab:red',
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec='tab:red', lw=0.4, alpha=0.85),
                zorder=6)
        if _plateaus:
            ax_prof.plot([], [], color='tab:red', lw=2.0, label='plateau (mediana)')
        ax_prof.set_xlabel('ascissa curvilinea lungo la slice (px nativi)')
        ax_prof.set_ylabel(r'$\delta$ (°)')
        ax_prof.set_ylim(-10, 370)
        ax_prof.set_yticks(np.arange(0, 361, 60))
        ax_prof.grid(alpha=0.3)
        ax_prof.legend(loc='lower right', fontsize=7, framealpha=0.9)
        ax_prof.set_title(
            fr'Profilo lungo slice a {SLICE_ANGLE_F:.0f}° '
            f'(ancora ({SLICE_ANCHOR_F[0]}, {SLICE_ANCHOR_F[1]}))', pad=6)

        if _fit is not None:
            ax_fit = axes[2]
            x_fit = _fit['x']; y_fit = _fit['y']; slope = _fit['slope']
            for xi, yi, lbl in zip(x_fit, y_fit, _fit['labels']):
                if lbl.endswith('L'):
                    marker, color = 'o', 'tab:blue'
                elif lbl.endswith('R'):
                    marker, color = 's', 'tab:orange'
                else:
                    marker, color = 'D', 'tab:green'
                ax_fit.plot(xi, yi, marker=marker, color=color, ms=6,
                            markeredgecolor='k', markeredgewidth=0.5,
                            linestyle='none', zorder=3)
                ax_fit.annotate(lbl, (xi, yi), xytext=(4, 3),
                                textcoords='offset points', fontsize=6)
            x_max = float(x_fit.max())
            xs = np.array([0.0, x_max + 0.3])
            ax_fit.plot(xs, slope * xs, 'k--', lw=1.0,
                        label=fr'$\delta = {slope:.2f}\,n$')
            n_unw = int(_fit.get('n_unwrapped', 0))
            if n_unw:
                ax_fit.plot(x_fit, _fit['y_raw'], 'x', color='gray',
                            ms=5, alpha=0.5, zorder=2)
            ax_fit.set_xlim(0, x_max + 0.5)
            ymax = max(float(y_fit.max()) * 1.05,
                       slope * (x_max + 0.3) * 1.05, 360.0)
            ax_fit.set_ylim(0, ymax)
            ax_fit.set_xlabel(r'numero di strati $n$')
            ax_fit.set_ylabel(r'$\delta$ unwrap (°)' if n_unw else r'$\delta$ (°)')
            ax_fit.grid(alpha=0.3)
            unw_note = f" (+{n_unw} unwrap)" if n_unw else ""
            ax_fit.set_title(
                fr'Fit $\delta = m\,n${unw_note}' '\n'
                fr'$m={slope:.2f}$°/strato, $R^2={_fit["r2"]:.3f}$',
                pad=6)
            handles = [
                _Line2D_F([], [], marker='o', color='tab:blue', linestyle='none',
                          markeredgecolor='k', markeredgewidth=0.5, label='lato sinistro (L)'),
                _Line2D_F([], [], marker='s', color='tab:orange', linestyle='none',
                          markeredgecolor='k', markeredgewidth=0.5, label='lato destro (R)'),
                _Line2D_F([], [], marker='D', color='tab:green', linestyle='none',
                          markeredgecolor='k', markeredgewidth=0.5, label='centro'),
                _Line2D_F([], [], color='k', linestyle='--', label='fit'),
            ]
            ax_fit.legend(handles=handles, loc='lower right', fontsize=6, framealpha=0.9)

        fig.suptitle(fr'Slice diagonale - {DATASET}, canale {_ch}', fontsize=11, y=1.02)
        fig.tight_layout()
        _pdf_path = os.path.join(_out_F, f"{_ch}_slice.pdf")
        polplot.save_and_show(fig, _pdf_path, show=SAVE_PLOTS)
        plt.close(fig)

        if _PLOTLY_OK_F:
            arc_native_html = _t_vals * DOWNSAMPLE_FACTOR
            figh = _go_F.Figure()
            figh.add_trace(_go_F.Scatter(
                x=arc_native_html[sl], y=_profile[sl] + _circ_std[sl],
                mode='lines', line=dict(width=0, color='gray'),
                showlegend=False, hoverinfo='skip'))
            figh.add_trace(_go_F.Scatter(
                x=arc_native_html[sl], y=_profile[sl] - _circ_std[sl],
                mode='lines', line=dict(width=0, color='gray'),
                fill='tonexty', fillcolor='rgba(150,150,150,0.25)',
                name='± σ circolare', hoverinfo='skip'))
            figh.add_trace(_go_F.Scatter(
                x=arc_native_html[sl], y=_profile[sl],
                mode='lines', line=dict(color='black', width=1.5),
                name='δ medio (banda)',
                hovertemplate='ascissa: %{x:.0f} px<br>δ: %{y:.2f}°<extra></extra>'))
            for p in _plateaus:
                figh.add_trace(_go_F.Scatter(
                    x=[p['arc_start'], p['arc_end']],
                    y=[p['delta_med'], p['delta_med']],
                    mode='lines', line=dict(color='red', width=3),
                    name=f"{p['label']}: {p['delta_med']:.1f}°",
                    hovertemplate=(f"<b>{p['label']}</b><br>"
                                   f"δ = {p['delta_med']:.2f}°<br>"
                                   f"σ = {p['delta_std']:.2f}°<extra></extra>"),
                    showlegend=False))
            figh.update_layout(
                title=(f"Slice diagonale - {DATASET}, canale {_ch} "
                       f"(λ = {_wavelength:.0f} nm)"),
                xaxis=dict(title='ascissa curvilinea (px nativi)'),
                yaxis=dict(title='δ (°)', range=[-10, 370],
                           tickvals=list(range(0, 361, 60))),
                width=1000, height=520,
                margin=dict(l=70, r=30, t=70, b=60),
                template='simple_white', hovermode='x unified')
            _html_path = os.path.join(_out_html_F, f"{_ch}_slice.html")
            figh.write_html(_html_path, include_plotlyjs='cdn', full_html=True)
            print(f"  HTML salvato: {_html_path}")

## H — Fit retardance vs strati + dispersione 1/λ²

Punti retardance per strato (3 canali RGB) + fit lineare attraverso l'origine + correzione dispersiva 1/λ². Solo `strati_v2`.

In [ ]:
if 'H' in active_analyses:
    from scipy.optimize import curve_fit as _curve_fit_H
    try:
        import plotly.graph_objects as _go_H
        _PLOTLY_OK_H = True
    except ImportError:
        _PLOTLY_OK_H = False

    STRATI_CSV = './strati_retardance.csv'
    _N_STRATI_DEFAULT = np.array([1, 2, 3, 4, 5, 4, 3, 2, 1])
    _DELTA_PLACEHOLDER = {
        'R': np.array([229.0, 463.0, 720.0, 959.0, 1205.0, 964.0, 720.0, 488.0, 238.0]),
        'G': np.array([274.0, 559.0, 825.0, 1129.0, 1415.0, 1140.0, 841.0, 544.0, 280.0]),
        'B': np.array([328.0, 652.0, 947.0, 1249.0, 1576.0, 1258.0, 991.0, 677.0, 336.0]),
    }

    def _load_strati_csv(path):
        if not os.path.exists(path):
            print(f"  WARN: {path} non trovato; uso placeholder pre-arctan2 "
                  "(valori da rimisurare, vedi TODO B2)")
            return _N_STRATI_DEFAULT.copy(), {k: v.copy() for k, v in _DELTA_PLACEHOLDER.items()}
        try:
            import csv as _csv
            with open(path, 'r') as f:
                reader = _csv.reader(f)
                header = next(reader, None)
                rows = [r for r in reader if r and not r[0].startswith('#')]
            if not rows:
                raise ValueError("CSV vuoto")
            n_col = 0
            cols = {h.strip(): i for i, h in enumerate(header)}
            n_strati = np.array([int(r[cols.get('n', n_col)]) for r in rows], dtype=int)
            delta = {}
            for ch in ('R', 'G', 'B'):
                if ch in cols:
                    delta[ch] = np.array([float(r[cols[ch]]) for r in rows], dtype=float)
            if not delta:
                raise ValueError("nessuna colonna R/G/B")
            print(f"  letti {len(rows)} punti retardance da {path}")
            return n_strati, delta
        except Exception as e:
            print(f"  WARN: lettura {path} fallita ({e}); uso placeholder")
            return _N_STRATI_DEFAULT.copy(), {k: v.copy() for k, v in _DELTA_PLACEHOLDER.items()}

    def _linear_zero(x, m): return m * x
    def _inverse(x, k): return k / x

    def _fmt_sci(value):
        if value == 0: return "0"
        exponent = int(np.floor(np.log10(abs(value))))
        coeff = value / (10 ** exponent)
        return rf"{coeff:.1f} \cdot 10^{{{exponent}}}"

    _n_strati, _delta_meas = _load_strati_csv(STRATI_CSV)

    _lambda_R = polcfg.get_channel_wavelength(WAVELENGTHS_CSV, 0)
    _lambda_G = polcfg.get_channel_wavelength(WAVELENGTHS_CSV, 1)
    _lambda_B = polcfg.get_channel_wavelength(WAVELENGTHS_CSV, 2)
    _lambdas = np.array([_lambda_R, _lambda_G, _lambda_B])
    print(f"  lambda RGB = {_lambda_R:.0f}, {_lambda_G:.0f}, {_lambda_B:.0f} nm")

    _out_H = os.path.join(OUTPUT_DIR, 'strati_fit')
    os.makedirs(_out_H, exist_ok=True)

    _colors_H = {'R': 'tab:red', 'G': 'tab:green', 'B': 'tab:blue'}
    _slopes = []

    fig, ax = plt.subplots(figsize=(3.35, 3.35))
    for ch in ('R', 'G', 'B'):
        if ch not in _delta_meas:
            print(f"  skip canale {ch}: dati assenti")
            _slopes.append(np.nan); continue
        y = _delta_meas[ch]
        popt, _ = _curve_fit_H(_linear_zero, _n_strati, y)
        slope = float(popt[0])
        _slopes.append(slope)
        ax.scatter(_n_strati, y, color=_colors_H[ch], s=15, zorder=3, alpha=0.8)
        x_fit_line = np.linspace(0, max(5.5, float(_n_strati.max()) + 0.5), 100)
        ax.plot(x_fit_line, _linear_zero(x_fit_line, slope),
                color=_colors_H[ch], linestyle='--', linewidth=1.2,
                label=rf'{ch}: $\delta_{ch} \approx {slope:.1f}^\circ/n$', zorder=2)
    ax.set_xlabel(r'Numero di strati $n$')
    ax.set_ylabel(r'Ritardo di fase srotolato $\delta_{unwrap}$ (°)')
    ax.set_xlim(0, max(5.5, float(_n_strati.max()) + 0.5))
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left')
    _pdf_lin = os.path.join(_out_H, 'fit_strati_linear.pdf')
    polplot.save_and_show(fig, _pdf_lin, show=SAVE_PLOTS)
    plt.close(fig)

    _slopes = np.array(_slopes)
    _valid_slopes = ~np.isnan(_slopes)
    if _valid_slopes.sum() >= 2:
        popt_lam, _ = _curve_fit_H(_inverse, _lambdas[_valid_slopes], _slopes[_valid_slopes])
        k = float(popt_lam[0])
        k_str = _fmt_sci(k)
    else:
        k = float('nan'); k_str = "n/a"
        print("  WARN: meno di 2 slope validi, fit 1/lambda non eseguito")

    fig, ax = plt.subplots(figsize=(3.35, 3.35))
    for i, (ch, color) in enumerate([('R', 'tab:red'), ('G', 'tab:green'), ('B', 'tab:blue')]):
        if not np.isnan(_slopes[i]):
            ax.scatter(_lambdas[i], _slopes[i], color=color, s=25, zorder=3, label=ch)
    if np.isfinite(k):
        x_lam = np.linspace(150, 1100, 500)
        ax.plot(x_lam, _inverse(x_lam, k), color='black', linestyle='--', linewidth=1.2,
                label=rf'Fit: $\delta(\lambda) = \frac{{{k_str}}}{{\lambda}}$', zorder=2)
    ax.set_xlabel(r"Lunghezza d'onda $\lambda$ (nm)")
    ax.set_ylabel(r'Ritardo specifico $\delta$ (°/strato)')
    ax.set_xlim(0, 1100)
    if _valid_slopes.any():
        ax.set_ylim(0, max(_slopes[_valid_slopes]) * 1.4)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper right')
    _pdf_lam = os.path.join(_out_H, 'fit_lambda_inverse.pdf')
    polplot.save_and_show(fig, _pdf_lam, show=SAVE_PLOTS)
    plt.close(fig)

    if _PLOTLY_OK_H:
        figh = _go_H.Figure()
        for ch, color_rgb in [('R', 'rgb(214,39,40)'), ('G', 'rgb(44,160,44)'),
                              ('B', 'rgb(31,119,180)')]:
            if ch not in _delta_meas:
                continue
            y = _delta_meas[ch]
            i = {'R': 0, 'G': 1, 'B': 2}[ch]
            slope = _slopes[i]
            figh.add_trace(_go_H.Scatter(
                x=_n_strati, y=y, mode='markers',
                marker=dict(color=color_rgb, size=8),
                name=f"{ch} (misurato)",
                hovertemplate=f'n=%{{x}}<br>δ=%{{y:.1f}}°<extra>{ch}</extra>'))
            if np.isfinite(slope):
                x_line = np.linspace(0, max(5.5, float(_n_strati.max()) + 0.5), 50)
                figh.add_trace(_go_H.Scatter(
                    x=x_line, y=slope * x_line, mode='lines',
                    line=dict(color=color_rgb, dash='dash', width=1.5),
                    name=f"{ch}: δ≈{slope:.1f}°/n", showlegend=True))
        figh.update_layout(
            title=f"Retardance vs numero di strati - {DATASET}",
            xaxis=dict(title='numero di strati n'),
            yaxis=dict(title='δ unwrap (°)'),
            template='simple_white', width=700, height=500,
            margin=dict(l=70, r=30, t=70, b=60))
        _html_lin = os.path.join(_out_H, 'fit_strati_linear.html')
        figh.write_html(_html_lin, include_plotlyjs='cdn', full_html=True)
        print(f"  HTML salvato: {_html_lin}")

        figh2 = _go_H.Figure()
        for i, (ch, color_rgb) in enumerate([('R', 'rgb(214,39,40)'),
                                              ('G', 'rgb(44,160,44)'),
                                              ('B', 'rgb(31,119,180)')]):
            if not np.isnan(_slopes[i]):
                figh2.add_trace(_go_H.Scatter(
                    x=[_lambdas[i]], y=[_slopes[i]], mode='markers',
                    marker=dict(color=color_rgb, size=12),
                    name=ch,
                    hovertemplate=f'λ=%{{x:.0f}} nm<br>δ=%{{y:.1f}}°/strato<extra>{ch}</extra>'))
        if np.isfinite(k):
            x_lam = np.linspace(150, 1100, 500)
            figh2.add_trace(_go_H.Scatter(
                x=x_lam, y=k / x_lam, mode='lines',
                line=dict(color='black', dash='dash'),
                name=f"fit k/λ, k={k:.2e}"))
        figh2.update_layout(
            title=f"Dispersione 1/λ - {DATASET}",
            xaxis=dict(title='λ (nm)', range=[0, 1100]),
            yaxis=dict(title='δ (°/strato)'),
            template='simple_white', width=700, height=500,
            margin=dict(l=70, r=30, t=70, b=60))
        _html_lam = os.path.join(_out_H, 'fit_lambda_inverse.html')
        figh2.write_html(_html_lam, include_plotlyjs='cdn', full_html=True)
        print(f"  HTML salvato: {_html_lam}")

## Strumenti opzionali

Celle gated dai flag `RUN_*` in cima al notebook (off-by-default).

### G0 — Stima centroidi spettrali RGB

Esegue `final_monochrome_approx` (legacy) e aggiorna `outputs/rgb_wavelengths.csv` + PDF spettrale.

In [ ]:
if RUN_SPECTRA:
    import pandas as pd

    os.makedirs('outputs', exist_ok=True)

    df_camera = pd.read_csv('spettri/Samsung-Galaxy-S22-Rear-Telephoto-Camera.csv')
    df_sorgente = pd.read_csv('spettri/rgb.csv', sep=';')

    lam_min = max(df_camera['wavelength'].min(), df_sorgente['lambda'].min())
    lam_max = min(df_camera['wavelength'].max(), df_sorgente['lambda'].max())
    lam = np.arange(np.ceil(lam_min), np.floor(lam_max) + 1, 1.0)

    sens_R = np.interp(lam, df_camera['wavelength'], df_camera['red'])
    sens_G = np.interp(lam, df_camera['wavelength'], df_camera['green'])
    sens_B = np.interp(lam, df_camera['wavelength'], df_camera['blue'])
    src = np.interp(lam, df_sorgente['lambda'], df_sorgente['rgb'])

    eff_R = sens_R * src
    eff_G = sens_G * src
    eff_B = sens_B * src

    SOGLIA = 0.3

    def _centroide(arr):
        peak_val = arr[np.argmax(arr)]
        mask = arr >= peak_val * SOGLIA
        area = float(np.sum(arr[mask]))
        if area == 0:
            return float(lam[np.argmax(arr)])
        return float(np.sum(lam[mask] * arr[mask]) / area)

    centroide_R = _centroide(eff_R)
    centroide_G = _centroide(eff_G)
    centroide_B = _centroide(eff_B)

    print(f"R centroide {centroide_R:.0f} nm")
    print(f"G centroide {centroide_G:.0f} nm")
    print(f"B centroide {centroide_B:.0f} nm")

    fig, axs = plt.subplots(5, 1, figsize=(10, 14), dpi=150, sharex=True)
    fig.subplots_adjust(hspace=0.15, top=0.95, bottom=0.08)
    fig.suptitle(f"Analisi Spettrale dei Canali RGB (Baricentro con soglia {SOGLIA*100:.0f}%)",
                 fontsize=16, fontweight='bold')
    fig.text(0.5, 0.02,
             'Dati Sensore (S22 proxy): Color Lab Eilat (github.io/Spectral-sensitivity-estimation-web)',
             ha='center', fontsize=9, color='dimgray')

    axs[0].plot(lam, sens_R, color='#d62728', label='Sensibilità R')
    axs[0].plot(lam, sens_G, color='#2ca02c', label='Sensibilità G')
    axs[0].plot(lam, sens_B, color='#1f77b4', label='Sensibilità B')
    axs[0].set_ylabel("Risposta Relativa", fontsize=10)
    axs[0].set_title("Sensibilità Spettrale del Sensore", loc='left', fontsize=12)
    axs[0].grid(True, linestyle=':', alpha=0.6)
    axs[0].legend(loc='upper right', fontsize=9)

    axs[1].plot(lam, src, color='black')
    axs[1].fill_between(lam, src, color='gray', alpha=0.2)
    axs[1].set_ylabel("Intensità", fontsize=10)
    axs[1].set_title("Spettro di Emissione della Sorgente", loc='left', fontsize=12)
    axs[1].grid(True, linestyle=':', alpha=0.6)

    ylim_max = max(eff_R.max(), eff_G.max(), eff_B.max()) * 1.15

    for ax, eff, color, centroide, label in [
        (axs[2], eff_R, '#d62728', centroide_R, 'R'),
        (axs[3], eff_G, '#2ca02c', centroide_G, 'G'),
        (axs[4], eff_B, '#1f77b4', centroide_B, 'B'),
    ]:
        ax.plot(lam, eff, color=color)
        ax.fill_between(lam, eff, where=(eff >= eff.max() * SOGLIA), color=color, alpha=0.3)
        ax.axvline(x=centroide, color='black', linestyle='--', alpha=0.8,
                   label=f'Centroide: {centroide:.0f} nm')
        ax.set_ylabel(f"Segnale {label}", fontsize=10)
        ax.set_ylim(0, ylim_max)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='upper right', fontsize=9)
    axs[4].set_xlabel(r"Lunghezza d'onda $\lambda$ (nm)", fontsize=12)

    pdf_path = 'outputs/Analisi_Spettrale_S24_RGB.pdf'
    fig.savefig(pdf_path, format='pdf', bbox_inches='tight')
    print(f"saved: {pdf_path}")

    csv_path = 'outputs/rgb_wavelengths.csv'
    pd.DataFrame({
        'canale': ['R', 'G', 'B'],
        'centroide': [int(round(centroide_R)), int(round(centroide_G)), int(round(centroide_B))],
    }).to_csv(csv_path, index=False)
    print(f"saved: {csv_path}")

    plt.show()


### I — Debugger pixel-per-pixel

Ispettore interattivo del fit Stokes pixel-per-pixel (animazione intensità vs angolo).

In [ ]:
if RUN_DEBUGGER:
    from matplotlib.animation import FuncAnimation

    _dbg_ch = CHANNELS_RGB[ACTIVE_CHANNELS[0]] if ACTIVE_CHANNELS else polcfg.DEFAULT_TARGET_CHANNEL_IDX
    pol.reset_saturation_accumulator()
    _dbg_angles, _dbg_stack = pol.load_rotation_sequence(
        POL_SUBFOLDER, _dbg_ch,
        downsample_factor=DOWNSAMPLE_FACTOR,
        invert_angles=False,
        dark_frame_path=DARK_FRAME_PATH,
    )
    if _dbg_stack is None or _dbg_angles is None:
        raise RuntimeError(f"Debugger: impossibile caricare {POL_SUBFOLDER}")
    _dbg_S0, _dbg_S1, _dbg_S2 = pol.calculate_linear_stokes(_dbg_angles, _dbg_stack)

    _N, _H, _W = _dbg_stack.shape
    _sel = [_W // 2, _H // 2]
    _angles_deg = np.degrees(_dbg_angles) / 2.0
    _plot_lim = max(180.0, float(_angles_deg.max()))

    _fig = plt.figure(figsize=(18, 6), num="Pixel debugger")
    _ax_img = _fig.add_subplot(131)
    _ax_plot = _fig.add_subplot(132)
    _ax_ell = _fig.add_subplot(133)

    _vmax = float(np.percentile(_dbg_stack, 99))
    _img_disp = _ax_img.imshow(_dbg_stack[0], cmap='gray', vmin=0, vmax=_vmax)
    _ax_img.set_title(f"Clicca un pixel  |  {_W}x{_H}  |  q per uscire")
    _ax_img.axis('off')
    _marker, = _ax_img.plot(_sel[0], _sel[1], 'r+', markersize=12, markeredgewidth=2)

    _meas, = _ax_plot.plot([], [], 'ro', label='Misurato', zorder=5)
    _fit_line, = _ax_plot.plot([], [], 'b-', linewidth=2,
                               label='0.5(S0 + S1 cos2A + S2 sin2A)', zorder=4)
    _cur, = _ax_plot.plot([], [], 'yo', markersize=10, markeredgecolor='black',
                          label='Frame corrente', zorder=10)
    _ax_plot.set_xlabel("Angolo analizzatore (deg)")
    _ax_plot.set_ylabel("Intensità (DN)")
    _ax_plot.legend(loc='upper right')
    _ax_plot.grid(True, linestyle='--', alpha=0.5)

    _ell, = _ax_ell.plot([], [], 'g-', linewidth=2, label='Ellisse')
    _axis_line, = _ax_ell.plot([], [], 'r--', linewidth=1.5, label='AoLP')
    _ax_ell.set_aspect('equal', adjustable='box')
    _ax_ell.grid(True, linestyle='--', alpha=0.5)
    _ax_ell.legend(loc='upper right')
    _ax_ell.set_xlabel("X intensity")
    _ax_ell.set_ylabel("Y intensity")

    def _update_pixel(x, y):
        _sel[0], _sel[1] = x, y
        _marker.set_data([x], [y])
        ints = _dbg_stack[:, y, x]
        _meas.set_data(_angles_deg, ints)
        s0, s1, s2 = float(_dbg_S0[y, x]), float(_dbg_S1[y, x]), float(_dbg_S2[y, x])
        xd = np.linspace(0, _plot_lim, 200)
        xr2 = np.deg2rad(xd * 2)
        yf = 0.5 * (s0 + s1 * np.cos(xr2) + s2 * np.sin(xr2))
        _fit_line.set_data(xd, yf)
        _ax_plot.set_title(f"Pixel ({x}, {y}) | S0={s0:.0f}, S1={s1:.0f}, S2={s2:.0f}")
        _ax_plot.set_xlim(-5, _plot_lim + 5)
        _ax_plot.set_ylim(0, max(float(ints.max()), float(yf.max())) * 1.1)

        s0s = s0 if s0 > 0 else 1e-8
        dolp = float(np.clip(np.sqrt(s1 ** 2 + s2 ** 2) / s0s, 0, 1))
        aolp = 0.5 * np.arctan2(s2, s1)
        A = np.sqrt(0.5 * s0s * (1 + dolp))
        B = np.sqrt(0.5 * s0s * (1 - dolp))
        t = np.linspace(0, 2 * np.pi, 150)
        xe = A * np.cos(t) * np.cos(aolp) - B * np.sin(t) * np.sin(aolp)
        ye = A * np.cos(t) * np.sin(aolp) + B * np.sin(t) * np.cos(aolp)
        _ell.set_data(xe, ye)
        _axis_line.set_data([-A * np.cos(aolp), A * np.cos(aolp)],
                            [-A * np.sin(aolp), A * np.sin(aolp)])
        lim = np.sqrt(s0s) * 1.1 + 1e-5
        _ax_ell.set_xlim(-lim, lim)
        _ax_ell.set_ylim(-lim, lim)
        _ax_ell.set_title(f"DoLP {dolp:.2%}  |  AoLP {np.degrees(aolp):.1f} deg")
        _fig.canvas.draw_idle()

    def _animate(frame):
        idx = frame % _N
        _img_disp.set_data(_dbg_stack[idx])
        x, y = _sel
        s0, s1, s2 = float(_dbg_S0[y, x]), float(_dbg_S1[y, x]), float(_dbg_S2[y, x])
        a2 = _dbg_angles[idx]
        cur_i = 0.5 * (s0 + s1 * np.cos(a2) + s2 * np.sin(a2))
        _cur.set_data([float(np.degrees(a2) / 2.0)], [cur_i])
        return _img_disp, _cur, _marker

    def _onclick(event):
        if event.inaxes == _ax_img and event.xdata is not None and event.ydata is not None:
            x = int(np.clip(event.xdata, 0, _W - 1))
            y = int(np.clip(event.ydata, 0, _H - 1))
            _update_pixel(x, y)

    def _onkey(event):
        if event.key == 'q':
            plt.close(_fig)

    _update_pixel(_W // 2, _H // 2)
    _anim = FuncAnimation(_fig, _animate, frames=range(20000),
                          interval=200, blit=False, cache_frame_data=False)
    _fig.canvas.mpl_connect('button_press_event', _onclick)
    _fig.canvas.mpl_connect('key_press_event', _onkey)
    plt.tight_layout()
    plt.show()


### FULL BATCH — 7 dataset × 3 canali × 9 parametri

Esegue AB su tutti i dataset attivi nel dispatcher; 189 PDF + 189 HTML in `OUTPUT_DIR/<dataset>/`.

In [ ]:
if RUN_FULL_BATCH:
    import time
    import traceback
    import importlib

    from polarimetro import io_raw as _polio
    from polarimetro import stokes as _polstokes
    from polarimetro import align as _polalign

    BATCH_DATASETS = list(ANALYSES_PER_DATASET.keys())
    BATCH_CHANNELS = [('R', 0), ('G', 1), ('B', 2)]

    plt.ioff()

    _t0 = time.time()
    _ok = []
    _failures = []
    _total = sum(len(BATCH_CHANNELS) for ds in BATCH_DATASETS if 'AB' in ANALYSES_PER_DATASET.get(ds, []))
    _done = 0

    for _ds in BATCH_DATASETS:
        _analyses = ANALYSES_PER_DATASET.get(_ds, [])
        if 'AB' not in _analyses:
            print(f"-- skip {_ds} (no AB)")
            continue

        _ds_root = f'./raw/{_ds}'
        _pol_dir = os.path.join(_ds_root, 'pol')
        _wav_dir = os.path.join(_ds_root, 'wav')
        _swap = polcfg.is_waveplate_swapped(_ds_root)

        for _ch_label, _ch_idx in BATCH_CHANNELS:
            _done += 1
            _banner = f"[{_done}/{_total}] {_ds} / {_ch_label}  swap={_swap}"
            print('\n' + '=' * len(_banner))
            print(_banner)
            print('=' * len(_banner))

            _polio._SATURATION_ACCUMULATOR = None
            _polstokes._WAV_INTENSITY_CACHE = None
            _polalign._POINCARE_BG_MASK_CACHE = None
            pol.reset_saturation_accumulator()

            try:
                _angles, _stack = pol.load_rotation_sequence(
                    _pol_dir, _ch_idx,
                    downsample_factor=DOWNSAMPLE_FACTOR,
                    invert_angles=False,
                    dark_frame_path=DARK_FRAME_PATH,
                )
                if _stack is None or _angles is None:
                    raise RuntimeError(f"load_rotation_sequence returned None for {_pol_dir}")

                _S0, _S1, _S2 = pol.calculate_linear_stokes(_angles, _stack)
                _lam = polcfg.get_channel_wavelength(WAVELENGTHS_CSV, _ch_idx)
                _S3 = pol.calculate_s3(
                    _wav_dir, _ch_idx,
                    downsample_factor=DOWNSAMPLE_FACTOR,
                    wavelength=_lam,
                    dark_frame_path=DARK_FRAME_PATH,
                )
                if _S3 is None:
                    raise RuntimeError(f"calculate_s3 returned None for {_wav_dir}")

                _bg_mask = pol.generate_background_mask(_S0)
                _S1, _S2 = pol.align_reference_frame(_S1, _S2, _bg_mask)
                _S1, _S3 = pol.align_poincare_ellipticity(
                    _S0, _S1, _S3, _bg_mask,
                    downsample_factor=DOWNSAMPLE_FACTOR,
                )
                _dolp, _aolp = pol.calculate_dolp_aolp(_S0, _S1, _S2)
                _delta, _theta = pol.calculate_retardance_and_fast_axis(
                    _S0, _S1, _S2, _S3, _bg_mask,
                    target_folder=_ds_root,
                )

                _outdir = os.path.join(OUTPUT_DIR, _ds)
                os.makedirs(_outdir, exist_ok=True)
                _sat = pol.get_saturation_mask(downsample_factor=DOWNSAMPLE_FACTOR)
                _poincare_mask = pol.get_poincare_bg_mask()
                _maps = {
                    'S0': _S0, 'S1': _S1, 'S2': _S2, 'S3': _S3,
                    'DoLP': _dolp, 'AoLP': _aolp,
                    'delta': _delta, 'theta': _theta,
                }
                _saved = 0
                for _name, _arr in _maps.items():
                    _arr_show = np.array(_arr, dtype=float)
                    if _sat is not None and _sat.shape == _arr_show.shape:
                        _arr_show = np.where(_sat, np.nan, _arr_show)
                    _fig = plt.figure(figsize=(4.2, 3.5))
                    _ax = _fig.add_subplot(111)
                    if _name in ('AoLP', 'theta'):
                        _im = _ax.imshow(_arr_show, cmap='twilight', vmin=-90, vmax=90)
                    elif _name == 'delta':
                        _im = _ax.imshow(_arr_show, cmap='twilight', vmin=0, vmax=360)
                    elif _name == 'DoLP':
                        _im = _ax.imshow(_arr_show, cmap='magma', vmin=0, vmax=1)
                    else:
                        _vmax = float(np.nanpercentile(_arr_show, 99)) if np.isfinite(_arr_show).any() else 1.0
                        _vmin = float(np.nanpercentile(_arr_show, 1)) if np.isfinite(_arr_show).any() else -1.0
                        _im = _ax.imshow(_arr_show, cmap='gray', vmin=_vmin, vmax=_vmax)
                    _ax.set_title(f"{_ds}/{_ch_label} {_name}")
                    _ax.axis('off')
                    _fig.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04)
                    _path = os.path.join(_outdir, f"{_ch_label}_{_name}.pdf")
                    _fig.savefig(_path, format='pdf', bbox_inches='tight', dpi=300)
                    plt.close(_fig)
                    _saved += 1

                _wav_cache = pol.get_wav_intensity_cache()
                _wav_mean = (_wav_cache / 2.0) if _wav_cache is not None else None
                _overlay = polplot.mask_overlay_rgb(_S0, _bg_mask, _poincare_mask, wav_mean=_wav_mean)
                _fig = plt.figure(figsize=(4.2, 3.5))
                _ax = _fig.add_subplot(111)
                _ax.imshow(_overlay)
                _ax.set_title(f"{_ds}/{_ch_label} mask")
                _ax.axis('off')
                _path = os.path.join(_outdir, f"{_ch_label}_mask.pdf")
                _fig.savefig(_path, format='pdf', bbox_inches='tight', dpi=300)
                plt.close(_fig)
                _saved += 1

                plt.close('all')
                _ok.append((_ds, _ch_label, _saved))
                print(f"  OK  PDF salvati: {_saved}")
            except Exception as _exc:
                plt.close('all')
                _failures.append((_ds, _ch_label, str(_exc)))
                print(f"  FAIL {_ds}/{_ch_label}: {_exc}")
                traceback.print_exc()

    plt.ion()
    _elapsed = time.time() - _t0
    _pdf_total = sum(n for _, _, n in _ok)
    print('\n' + '=' * 40)
    print(f"Tempo totale: {_elapsed/60:.1f} min")
    print(f"PDF generati: {_pdf_total}")
    print(f"OK: {len(_ok)}/{_total}  FAIL: {len(_failures)}")
    if _failures:
        print("Falliti:")
        for _ds, _ch, _err in _failures:
            print(f"  - {_ds}/{_ch}: {_err}")
